# F6-svd-spectral — Session 4: The Frobenius Norm and Low-Rank
Approximation

*One class session, roughly 85 minutes — the opening session of this
double unit's second sitting. Prerequisites: Sessions 1–3 (especially
the SVD atom sum and orthonormality of the singular vector families).*

**This session:** a single number for "how big is a matrix" — the
**Frobenius norm** — and the identity that makes it the SVD's native
ruler: $\lVert W \rVert_F^2 = \sum_i \sigma_i^2$, **derived
component-wise** from the SVD expansion (this derivation is practice
problem p15's proof, worked in full).
Then the unit's punchline: **truncating** the SVD to its top $r$ atoms
gives a rank-$r$ approximation $W_r$, its error obeys the exact
identity $\lVert W - W_r \rVert_F^2 = \sum_{i > r} \sigma_i^2$
(p16's proof), no rank-$r$ matrix does better (Eckart–Young, stated),
and the storage arithmetic explains why anyone truncates in the first
place.
Plus the error-vs-$r$ curve and a fully worked normal-form MC.

Try every checkpoint by hand first, then verify with NumPy.
Answers are collected at the end of this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 20260804

## 1. The Frobenius Norm: a Matrix's Length

**Definition.** For any matrix $A\ (n, m)$,

$$\lVert A \rVert_F \;=\; \sqrt{\sum_{i}\sum_{j} A_{ij}^2}$$

— square every entry, add them all up, take the root.
It is exactly F2's vector norm applied to the matrix *flattened into
one long vector*; nothing about rows-vs-columns survives, only the
entries.

**Hand example.**
$A = \begin{pmatrix} 2 & -1 \\ 0 & 2 \\ 1 & -2 \end{pmatrix}$:
$\lVert A \rVert_F^2 = 4 + 1 + 0 + 4 + 1 + 4 = 14$, so
$\lVert A \rVert_F = \sqrt{14} \approx 3.742$.

**Idioms.** The from-entries route is `np.sqrt((A * A).sum())`; and
`np.linalg.norm(A)` — with **no** `ord` argument — computes exactly
the Frobenius norm for a 2-D input.
Two facts to keep straight:

- $\lVert A \rVert_F^2$ (the *squared* norm — additive, the quantity
  identities are stated in) vs $\lVert A \rVert_F$ (the norm — the
  quantity error budgets are usually stated in).
  Track which one a formula wants.
- $\lVert cA \rVert_F = |c| \cdot \lVert A \rVert_F$ and
  $\lVert A \rVert_F = \lVert A^{\mathsf T} \rVert_F$ — immediate from
  the definition (same entries).

In [ ]:
A = np.array([[2., -1.], [0., 2.], [1., -2.]])

fro_hand = np.sqrt((A * A).sum())
fro_lib = np.linalg.norm(A)
print("from entries:", fro_hand, " np.linalg.norm:", fro_lib)
print("sqrt(14)    :", np.sqrt(14.))
print("scale check |−3A| = 3|A|:",
      np.abs(np.linalg.norm(-3 * A) - 3 * fro_lib))

### Checkpoint 1

1. Compute $\lVert B \rVert_F$ by hand for
   $B = \begin{pmatrix} 1 & 2 & 2 \\ 0 & 3 & 0 \end{pmatrix}$.
2. Why is $\lVert A \rVert_F = 0$ possible only for the zero matrix?
3. Session 2 called $\mathrm{diag}(G)$ of a Gram matrix
   $G = WW^{\mathsf T}$ the squared row lengths.
   Express $\lVert W \rVert_F^2$ in terms of $G$'s diagonal.

## 2. The Master Identity: $\lVert W \rVert_F^2 = \sum_i \sigma_i^2$

The Frobenius norm looks like it belongs to the *entries*; the SVD
lives in *directions and stretches*.
This identity says they measure the same thing — and its proof is a
two-step component computation you will reproduce as practice p15.
Every move is F2/F3 material.

**Claim A (entry formula).**
From the SVD atom sum $W = \sum_k \sigma_k u_k v_k^{\mathsf T}$, entry
$(i, j)$ reads

$$W_{ij} = \sum_k \sigma_k \, u_k[i] \, v_k[j]$$

(each atom contributes its $(i,j)$ entry $u_k[i]v_k[j]$, scaled by
$\sigma_k$ — F3's outer-product entry rule).

**Claim B (square and swap).**
Squaring and summing over all entries, then swapping the (finite) sums:

$$
\lVert W \rVert_F^2
= \sum_{i,j} \Bigl(\sum_k \sigma_k u_k[i] v_k[j]\Bigr)
             \Bigl(\sum_\ell \sigma_\ell u_\ell[i] v_\ell[j]\Bigr)
= \sum_{k,\ell} \sigma_k \sigma_\ell
  \underbrace{\Bigl(\sum_i u_k[i]\,u_\ell[i]\Bigr)}_{u_k \cdot\, u_\ell}
  \underbrace{\Bigl(\sum_j v_k[j]\,v_\ell[j]\Bigr)}_{v_k \cdot\, v_\ell}.
$$

The entry sums have reorganized themselves into **dot products** of
singular vectors.

**Claim C (orthonormality kills the cross terms).**
$u_k \cdot u_\ell = 0$ for $k \ne \ell$ and $= 1$ for $k = \ell$ (unit
vectors, mutually perpendicular — F2), and likewise for the $v$'s.
Every cross term $k \ne \ell$ dies; every diagonal term survives with
coefficient $1 \cdot 1$:

$$\lVert W \rVert_F^2 = \sum_k \sigma_k^2. \qquad\blacksquare$$

**Why you should care** (beyond the exam): the squared Frobenius norm
is the matrix's total "content", and the SVD splits that content into
per-direction parcels $\sigma_k^2$ — with nothing lost and nothing
double-counted, *because* the directions are perpendicular.
Truncation (Section 3) is then just: keep the biggest parcels.

In [ ]:
rng = np.random.default_rng(SEED)
W6 = rng.normal(0, 1, (6, 4))
s6 = np.linalg.svd(W6, compute_uv=False)   # values only — handy variant

frob_sq = (W6**2).sum()
sig_sq = (s6**2).sum()
print("||W||_F^2 from entries:", frob_sq)
print("sum sigma_k^2         :", sig_sq)
print("gap:", abs(frob_sq - sig_sq))

(New idiom in passing: `np.linalg.svd(W, compute_uv=False)` returns
just the singular values — enough whenever only $\sigma$'s are
needed.)

### Checkpoint 2

1. In Claim B, the double sum over $(i, j)$ of a product of two
   $k$-sums became a double sum over $(k, \ell)$ of a product of an
   $i$-sum and a $j$-sum.
   Which elementary property of finite sums allows the swap, and why
   does the $i$-sum contain only $u$'s while the $j$-sum contains only
   $v$'s?
2. Where exactly would Claim C fail if the $u_k$ were unit vectors but
   *not* mutually perpendicular?
3. Use the identity to re-answer Session 3's Checkpoint 2.1: the rank-1
   matrix with entries $(1, 2; 2, 4)$ has $\sigma_1 = 5$ — verify
   against the entry sum.

## 3. Truncation: Keep the Top $r$ Atoms

The SVD orders its atoms by importance ($\sigma$'s descending).
The **rank-$r$ truncation** keeps the first $r$ and drops the rest:

$$W_r = \sum_{k \le r} \sigma_k u_k v_k^{\mathsf T}
      = U[:, :r] \;\mathrm{diag}(s[:r])\; V^{\mathsf T}[:r, :].$$

Slicing recipe, straight off the thin SVD's arrays: first $r$ columns
of `U`, first $r$ values of `s`, first $r$ **rows** of `Vt`.
By construction $\operatorname{rank}(W_r) \le r$ — it is a sum of $r$
rank-1 atoms (F3's rank bound for atom sums).

In [ ]:
def truncate(U, s, Vt, r):
    return U[:, :r] @ np.diag(s[:r]) @ Vt[:r, :]


U6, s6_, Vt6 = np.linalg.svd(W6, full_matrices=False)
for r in [0, 1, 2, 4]:
    Wr = truncate(U6, s6_, Vt6, r)
    err = np.linalg.norm(W6 - Wr)
    print(f"r = {r}:  ||W - W_r||_F = {err:.6f}")
print("(r = 4 = full rank: keeping every atom reproduces W)")

Note the two edge cases the loop just demonstrated: $r = 0$ keeps
nothing ($W_0 = 0$, error $= \lVert W \rVert_F$), and $r = \min(n, d)$
keeps everything (error machine-zero).
Everything interesting lives in between — how *fast* the error falls
is the subject of the rest of this session.

### Checkpoint 3

1. In the slicing recipe, why rows `[:r]` of `Vt` but columns `[:, :r]`
   of `U`?
2. What is $W_1$ for the diagonal matrix
   $\mathrm{diag}(5, 3, 1)$?
   (No code — think atoms.)
3. If two matrices $W$ and $3W$ are truncated at the same $r$, how do
   their truncations relate?

## 4. The Error Identity: $\lVert W - W_r \rVert_F^2 = \sum_{i>r}
\sigma_i^2$

What did truncation cost?
An *exact* accounting, no bounds needed — and the proof is one clean
application of Section 2 (this is practice p16's proof, worked here).

**Step 1 (the difference is a tail of atoms).**

$$W - W_r = \sum_{k > r} \sigma_k u_k v_k^{\mathsf T}$$

— subtracting the kept atoms from the full sum leaves the dropped
ones.

**Step 2 (the tail is itself an SVD-shaped sum).**
The vectors $u_{r+1}, \dots$ are orthonormal (they were orthonormal in
the full family; a sub-family stays orthonormal), likewise the tail
$v$'s, and the tail $\sigma$'s are non-negative and descending.
So $W - W_r$ is a matrix *given in SVD form*, whose singular values
are exactly $(\sigma_{r+1}, \sigma_{r+2}, \dots)$.

**Step 3 (apply the master identity to the tail).**

$$\lVert W - W_r \rVert_F^2 = \sum_{k > r} \sigma_k^2.
\qquad\blacksquare$$

Read it as bookkeeping: the total content $\sum_k \sigma_k^2$ splits
into kept parcels ($k \le r$) and dropped parcels ($k > r$); the
squared error is *precisely* the dropped content, to machine
precision, at every $r$:

In [ ]:
print("r  |  direct ||W - W_r||_F  |  sqrt(tail sum)  |  gap")
for r in range(5):
    direct = np.linalg.norm(W6 - truncate(U6, s6_, Vt6, r))
    formula = np.sqrt((s6_[r:]**2).sum())
    print(f"{r}  |  {direct:.12f}      |  {formula:.12f}  |  "
          f"{abs(direct - formula):.1e}")

### Checkpoint 4

1. $\sigma = (10, 6, 3, 1)$.
   Compute $\lVert W - W_2 \rVert_F$ exactly (by hand), and the
   *relative* error $\lVert W - W_2 \rVert_F / \lVert W \rVert_F$.
2. In Step 2, why does the sub-family $u_{r+1}, u_{r+2}, \dots$ remain
   orthonormal?
   And why must the identity be applied to the *tail's* singular
   values rather than $W$'s?
3. A dataset's singular values satisfy $\sigma_k = 0$ for all $k > 7$.
   What is $\lVert W - W_7 \rVert_F$, and what does that say about
   $W$'s rank?

## 5. Eckart–Young: Truncation Is Unbeatable (Stated)

The truncation $W_r$ is not just *a* rank-$r$ approximation — it is
the best one:

> **Fact (Eckart–Young; stated, optimality not proved — the error
> identity above IS derived and is the part you compute with).**
> For every matrix $B$ of rank at most $r$:
> $$\lVert W - B \rVert_F \;\ge\; \lVert W - W_r \rVert_F
> = \sqrt{\textstyle\sum_{i>r} \sigma_i^2}.$$
> No rank-$r$ matrix — however cleverly built, by any method —
> beats the top-$r$ SVD truncation in Frobenius error.

So "best rank-$r$ approximation" is a solved problem with a
constructive answer, and the error floor is readable off the spectrum
*before you build anything*.
A quick empirical taste — random rank-2 competitors against $W_2$:

In [ ]:
opt = np.linalg.norm(W6 - truncate(U6, s6_, Vt6, 2))
print("optimal rank-2 error:", opt)

rng = np.random.default_rng(SEED)
best_random = np.inf
for _ in range(2000):
    B = rng.normal(0, 1, (6, 2)) @ rng.normal(0, 1, (2, 4))  # rank <= 2
    best_random = min(best_random, np.linalg.norm(W6 - B))
print("best of 2000 random rank-2 tries:", best_random)
print("floor respected:", best_random >= opt)

Two thousand random attempts, none below the floor — as the theorem
guarantees for *all* attempts.

### Checkpoint 5

1. A colleague claims a rank-3 approximation of some $W$ with
   Frobenius error $1.9$, where $W$'s singular values are
   $(9, 5, 2, 2, 1)$.
   Use the floor to judge the claim.
2. Eckart–Young compares $W_r$ against *all* rank-$\le r$ matrices.
   Where does the competitor construction in the code guarantee rank
   $\le 2$?
   (F3 rank fact.)

## 6. Why Truncate? The Storage Arithmetic

Storing $W\ (n, m)$ costs $n \cdot m$ floats.
Storing the truncated form — $r$ triples
$(\sigma_k, u_k, v_k)$ — costs

$$r \cdot (n + m + 1) \text{ floats}$$

($n$ for each $u_k$, $m$ for each $v_k$, $1$ for each $\sigma_k$).
When $r \ll \min(n, m)$ this is a *massive* discount:

| $W$ | full | rank-$r$ form | ratio |
|---|---|---|---|
| $(30, 20)$ | $600$ | $r = 5$: $255$ | $42.5\%$ |
| $(200, 40)$ | $8000$ | $r = 5$: $1205$ | $15.1\%$ |
| $(1000, 1000)$ | $10^6$ | $r = 20$: $40020$ | $4.0\%$ |

The trade on offer, then: pay $\sqrt{\sum_{i>r}\sigma_i^2}$ in
Frobenius error, receive a storage bill of $r(n{+}m{+}1)$ instead of
$nm$.
Whether that trade is good depends entirely on how fast the spectrum
decays — which is why practitioners *plot* it, next section.

### Checkpoint 6

1. For a $(64, 48)$ matrix, at which $r$ does the rank-$r$ form stop
   being cheaper than full storage?
   (Solve $r(n + m + 1) < nm$.)
2. A friend stores the truncation as the *assembled* matrix $W_r$
   ($n \cdot m$ floats) "to keep things simple".
   What did the friend just pay for, and receive?

## 7. The Error-vs-$r$ Curve

For a matrix with genuine low-rank structure the identity turns the
spectrum into a *planning tool*: compute the SVD once, then read off —
before building anything — what every possible truncation costs.
The demo matrix below is built with five planted strong directions
(strengths $12, 8, 5, 3, 2$) plus low-level noise, a seeded stand-in
for real structured data tables.

In [ ]:
rng = np.random.default_rng(SEED)
n, m = 30, 20
strengths = np.array([12., 8., 5., 3., 2.])
Uraw = rng.normal(0, 1, (n, 5))
Vraw = rng.normal(0, 1, (m, 5))
Un = Uraw / np.sqrt((Uraw**2).sum(axis=0))     # unit columns (F2 normalize)
Vn = Vraw / np.sqrt((Vraw**2).sum(axis=0))
Wd = (Un * strengths) @ Vn.T + 0.10 * rng.normal(0, 1, (n, m))

Ud, sd, Vtd = np.linalg.svd(Wd, full_matrices=False)
print("top 8 singular values:", np.round(sd[:8], 4))
print("||W||_F:", np.linalg.norm(Wd))

The spectrum tells the story before any plot: five values that echo the
planted strengths, then a cliff down to a flat noise floor around
$0.6$.
Now the curve — absolute and relative error at every $r$, straight
from the identity (no truncations actually built):

In [ ]:
fro = np.linalg.norm(Wd)
rs = np.arange(0, 21)
errs = np.array([np.sqrt((sd[r:]**2).sum()) for r in rs])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(rs, errs, marker="o", ms=3)
axes[0].set_xlabel("r (atoms kept)")
axes[0].set_ylabel("||W - W_r||_F")
axes[0].set_title("absolute error vs r")
axes[1].plot(rs, errs / fro, marker="o", ms=3)
axes[1].axhline(0.15, ls="--", lw=1)
axes[1].set_xlabel("r (atoms kept)")
axes[1].set_ylabel("relative error")
axes[1].set_title("relative error vs r (dashed: 15% budget)")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

for r in [1, 2, 3, 4, 5, 6, 8]:
    print(f"r = {r}: relative error = {errs[r] / fro:.4f}")
r_budget = int(np.argmax(errs / fro <= 0.15))
print("smallest r meeting a 15% relative-error budget:", r_budget)

Reading the curve is a skill the exam tests:

- **The elbow.** The error falls steeply while real structure is being
  captured ($r \le 5$ here) and then flattens — atoms beyond the elbow
  buy almost nothing, because they carry noise-floor parcels.
- **Budget reads.** "Smallest $r$ with relative error $\le 15\%$" is a
  threshold scan of the curve (here: $r = 5$, matching the planted
  structure).
- **Diminishing, never negative, returns.** The curve is always
  non-increasing (each extra atom removes a $\sigma^2 \ge 0$ parcel
  from the tail) — a *rising* segment in an error-vs-$r$ plot is
  always a bug, usually the identity applied with `s[:r]` and `s[r:]`
  swapped.

### Checkpoint 7

1. From the printed relative errors: what would the budget answer be
   for a $40\%$ budget?
   For a $5\%$ budget, roughly where would you expect the answer to
   land, and why "roughly" — what controls it there?
2. Why is computing the whole curve from the identity *cheaper* than
   building each $W_r$ and measuring directly — and by how much,
   asymptotically in one sentence?

## 8. Worked Exam-Style Example: the Error Identity in Normal Form

---

**Worked exam-style example 4 (multiple choice, numeric normal form).**

> A matrix $W$ has singular values $\sigma = (8, 4, 2, 2, 1)$.
> Let $e = \lVert W - W_2 \rVert_F$ be the rank-2 truncation error and
> set $s = e^2 - 10$.
> Your answer can be written as $s = \pm m$ with $m$ a non-negative
> integer.
> What is the value of $2m - 1$ if $s \ge 0$, or $2m$ if $s < 0$?
>
> A. 2  B. 10  C. 14  D. 29  E. 139
>
> Reasoning is not required.

*Solution, step by step.*

1. **Which parcels drop.** Rank-2 truncation keeps
   $\sigma_1, \sigma_2 = 8, 4$; it drops $\sigma_3, \sigma_4, \sigma_5
   = 2, 2, 1$.
2. **The identity.** $e^2 = \sum_{i > 2} \sigma_i^2
   = 4 + 4 + 1 = 9$.
3. **Target.** $s = 9 - 10 = -1$.
4. **Normal form.** $m = 1$, branch $s < 0$.
5. **Decode.** $2m = 2$ → **A**.

Distractor forensics — each wrong option is a real mistake decoded:
drop only $\sigma_4, \sigma_5$ (off-by-one in "rank 2"):
$e^2 = 5,\ s = -5 \to 10$ = **B**; use $e$ instead of $e^2$:
$s = 3 - 10 = -7 \to 14$ = **C**; drop one atom
($e^2 = 16 + 4 + 4 + 1 = 25,\ s = 15 \to 29$) = **D**; sum the *kept*
parcels ($e^2 = 80,\ s = 70 \to 139$) = **E**.
Every option "decodes cleanly" — only the correct accounting of
kept-vs-dropped parcels selects A.

In [ ]:
sig = np.array([8., 4., 2., 2., 1.])
e2 = (sig[2:]**2).sum()
s_val = e2 - 10
m = abs(int(round(s_val)))
decoded = 2 * m - 1 if s_val >= 0 else 2 * m
print("e^2 =", e2, " s =", s_val, " m =", m,
      " decoded =", decoded, "-> option A")

### Checkpoint 8

1. Same register: $\sigma = (6, 3, 2, 1)$, $e = \lVert W - W_1
   \rVert_F$, $s = e^2 - 20$, same decode rule.
   Work it end to end.
2. Which two mistakes from the distractor forensics would a habit of
   writing "$e^2 = \sum_{i>r} \sigma_i^2$, with $i > r$ meaning
   *dropped*" have prevented?

## 9. Common Pitfalls IV

**Pitfall 1: error vs squared error.**
The identity is stated in $\lVert \cdot \rVert_F^2$; budgets are
usually stated in $\lVert \cdot \rVert_F$.
Mixing them inflates or deflates answers quadratically — the worked
MC's option C is exactly this slip.
Fix: write the superscript ${}^2$ (or its absence) into every line of
working.

**Pitfall 2: kept vs dropped.**
`(s[:r]**2).sum()` is the kept content, `(s[r:]**2).sum()` the dropped
— the error identity wants **dropped**.
The slice notation is treacherously symmetric; the worked MC's option
E is the swap.
Fix: sanity-anchor at the ends — $r = 0$ must give
$\lVert W \rVert_F^2$, $r = \min(n,m)$ must give $0$.

**Pitfall 3: relative error without the right denominator.**
"Relative to $W$" means dividing by $\lVert W \rVert_F$ — computed
from *all* parcels.
Dividing by the kept content, or by $\sigma_1$, gives plausible-looking
nonsense.

**Pitfall 4: reassembling to measure.**
Building $W_r$ explicitly to compute its error is legitimate (and a
good cross-check — Section 4's table) but wasteful in a loop; the
identity reads the whole curve off the spectrum.
On the exam, when a part says "compute the error for every $r$",
the identity *is* the intended route.

**Pitfall 5: `np.linalg.norm`'s other personalities.**
Bare `np.linalg.norm(A)` on a 2-D array is Frobenius — but
`np.linalg.norm(A, 2)` is a *different* matrix norm (the largest
singular value!), and on a 1-D input the same bare call is the vector
norm.
Fix in this course: bare call on 2-D for Frobenius, or the
from-entries route when any doubt exists.

In [ ]:
print("Frobenius (bare)     :", np.linalg.norm(Wd))
print("largest sigma (ord=2):", np.linalg.norm(Wd, 2))
print("sigma_1 for reference:", sd[0])

### Checkpoint 9

1. A script reports "rank-3 error $= 45.1$" for a matrix with
   $\lVert W \rVert_F = 16$.
   Which pitfall almost certainly fired, and what was probably
   computed?
2. Predict without running: for the demo matrix `Wd`, is
   `np.linalg.norm(Wd, 2)` bigger or smaller than
   `np.linalg.norm(Wd)`, and can they ever be equal for a nonzero
   matrix?

*(Sessions 4–5 form the second sitting; take a break here if you are
running both in one day.)*

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. $\lVert B \rVert_F^2 = 1 + 4 + 4 + 0 + 9 + 0 = 18$, so
   $\lVert B \rVert_F = \sqrt{18} = 3\sqrt2 \approx 4.243$.
2. A sum of squares is $0$ only when every term is $0$ — every entry
   must vanish.
3. $\lVert W \rVert_F^2 = \sum_i \lVert w_i \rVert^2
   = \sum_i G_{ii}$ — the sum of $G$'s diagonal entries (a row-wise
   accounting of the same sum over all squared entries).

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. Finite sums can be reordered/distributed freely (commutativity +
   distributivity).
   After expanding the product, each term factors as
   $\sigma_k \sigma_\ell \cdot u_k[i]u_\ell[i] \cdot v_k[j]v_\ell[j]$
   — the $i$-dependence sits only in the $u$ factors and the
   $j$-dependence only in the $v$ factors, so summing over $i$ and $j$
   factorizes.
2. The cross terms $k \ne \ell$ would keep the value
   $\sigma_k \sigma_\ell (u_k \cdot u_\ell)(v_k \cdot v_\ell) \ne 0$,
   and the total would be $\sum_k \sigma_k^2$ *plus* uncontrolled
   cross contributions.
3. Entry sum $= 1 + 4 + 4 + 16 = 25 = 5^2 + 0^2$. ✓

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. Because `Vt` stores $V^{\mathsf T}$: the right singular vectors are
   its ROWS, so the first $r$ vectors are `Vt[:r, :]`; `U` stores the
   left vectors as COLUMNS, hence `U[:, :r]`.
2. $\mathrm{diag}(5, 0, 0)$ — the top atom is the $\sigma_1 = 5$
   stretch along the first axis; the atoms here are
   $\sigma_k\,e_k e_k^{\mathsf T}$.
3. $(3W)_r = 3 \cdot W_r$: scaling multiplies every $\sigma_k$ by 3
   but reorders nothing, so the same atoms are kept, each scaled by 3.

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. $\lVert W - W_2 \rVert_F = \sqrt{3^2 + 1^2} = \sqrt{10}
   \approx 3.162$;
   $\lVert W \rVert_F = \sqrt{100 + 36 + 9 + 1} = \sqrt{146}$, so the
   relative error is $\sqrt{10/146} \approx 0.262$.
2. Orthonormality is a pairwise property (each pair of distinct
   vectors perpendicular, each vector unit) — deleting some vectors
   deletes constraints, never violates the surviving ones.
   The identity computes the norm of *the matrix at hand* from *its
   own* singular values; the tail matrix's singular values are the
   tail $\sigma$'s, not all of $W$'s.
3. $0$ — and $\operatorname{rank}(W) \le 7$: the truncation at the
   rank reproduces the matrix exactly.

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. The floor for $r = 3$ is $\sqrt{2^2 + 1^2} = \sqrt5 \approx 2.236$.
   The claim of $1.9 < 2.236$ contradicts Eckart–Young — the claim is
   wrong (an arithmetic slip, a different norm, or a rank higher than
   stated).
2. $B = XY$ with $X\ (6, 2)$, $Y\ (2, 4)$:
   $\operatorname{rank}(XY) \le \min(\operatorname{rank}X,
   \operatorname{rank}Y) \le 2$ — F3's product-rank bound.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. $r \cdot 113 < 3072 \iff r \le 27$: from $r = 28$ up
   ($28 \cdot 113 = 3164 > 3072$) the "compressed" form is bigger.
2. Paid: the approximation error $\sqrt{\sum_{i>r}\sigma_i^2}$.
   Received: nothing — assembled $W_r$ costs the same $n m$ floats as
   $W$.
   The storage win lives entirely in keeping the factors.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. $40\%$: $r = 2$ (first $r$ with relative error $\le 0.40$ —
   $r = 2$ gives $0.396$).
   $5\%$: beyond the elbow the tail is noise-floor parcels of nearly
   equal size, so the answer is controlled by how many noise parcels
   must be shaved — expect a much larger $r$ (double digits), not a
   structural read.
2. The identity route: one SVD, then each $r$ costs an $O(\min(n,m))$
   tail sum; the direct route rebuilds an $(n, m)$ matrix per $r$ —
   $O(nm\,r)$-ish work each — so the curve costs roughly a factor of
   the matrix size more.

</details>

<details><summary><b>Checkpoint 8</b></summary>

1. $e^2 = 9 + 4 + 1 = 14$; $s = -6$; $m = 6$; branch $s < 0$: decode
   $2m = 12$.
2. Option D (off-by-one: dropping an atom that was kept) and option E
   (summing kept instead of dropped) — both are kept/dropped
   accounting errors that the "$i > r$ means dropped" habit pins.

</details>

<details><summary><b>Checkpoint 9</b></summary>

1. Pitfall 2 (kept-vs-dropped swap): $45.1$ exceeds the whole matrix's
   norm $16$, impossible for a truncation error… and indeed
   $45.1 \approx$ the *kept* content masquerading as error — here
   likely `(s[:3]**2).sum()` reported without even the square root
   (Pitfall 1 stacking on top).
2. Smaller-or-equal: $\sigma_1 \le \sqrt{\sum_k \sigma_k^2}$ always.
   Equal exactly when the sum has one nonzero term — a rank-1 matrix.

</details>